# Your First LLM-Powered Tool

Starter notebook. Fill in the TODOs and **run every cell** before committing.
Do **not** commit your API key — load it from `.env`.


In [61]:
# Setup
import os
from dotenv import load_dotenv

# Load the variables from your .env file into the environment
load_dotenv()

import os, json

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY (see .env.example)"

# client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemma-4-26b-a4b-it"

## Task 1 — First calls and the sampling dial

Make a working call (print response **and token usage**), then send the same prompt 3× at temp ~0.1 and 3× at temp ~1.0.


In [66]:
# TODO: make a call with a system + user message; print response.text and the usage metadata.
# TODO: loop temperatures [0.1, 1.0] x 3 runs each and print outputs.
import time
from google import genai
from google.genai import types

# Initialize client
client = genai.Client()

sample_ticket = "I was charged twice for my subscription this month. Please refund the duplicate charge."
system_prompt = "You are a concise support assistant. Classify the user message into one of these labels: billing, bug, feature_request, other. Only return the label."

# Loop through both settings
for temp in [0.1, 1.0]:
    print(f"\n=== Testing Temperature: {temp} ===")
    
    for i in range(3):
        # Configure settings
        config = types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=temp,
        )
        
        # Call API
        response = client.models.generate_content(
            model='gemma-4-26b-a4b-it',
            contents=sample_ticket,
            config=config
        )
        
        # Print neat results
        print(f"Run #{i+1} | Output: {response.text.strip()}")
        tokens = response.usage_metadata
        print(f"        | Tokens -> Input: {tokens.prompt_token_count} | Output: {tokens.candidates_token_count}")
        print("-" * 30)
        
        time.sleep(12)


=== Testing Temperature: 0.1 ===
Run #1 | Output: billing
        | Tokens -> Input: 51 | Output: 1
------------------------------
Run #2 | Output: billing
        | Tokens -> Input: 51 | Output: 1
------------------------------
Run #3 | Output: billing
        | Tokens -> Input: 51 | Output: 1
------------------------------

=== Testing Temperature: 1.0 ===
Run #1 | Output: billing
        | Tokens -> Input: 51 | Output: 1
------------------------------
Run #2 | Output: billing
        | Tokens -> Input: 51 | Output: 1
------------------------------
Run #3 | Output: billing
        | Tokens -> Input: 51 | Output: 1
------------------------------


**What changed, and why?**  
What changed: Across all 6 runs (3 at temperature 0.1 and 3 at temperature 1.0), the model returned the exact same output: billing. The token counts also remained completely identical (51 input tokens, 1 output token).

Why: Temperature controls the randomness of the model's token selection. At a low temperature (0.1), the model behaves deterministically by strictly choosing the highest-probability token ("greedy decoding"). At a high temperature (1.0), the model is allowed to explore lower-probability tokens. However, because this specific ticket is incredibly clear and unambiguous, the probability for the token billing was likely close to 100%. Even with the randomness turned up at 1.0, the alternatives were too weak to be chosen, resulting in consistent outputs.


## Task 2 — Prompt-pattern bake-off

Classify the tickets three ways (zero-shot, few-shot, chain-of-thought) into `billing / bug / feature_request / other`. Build a comparison table. Record prompts + verdict in `prompts.md`.


In [ ]:
with open("tickets.json") as f:
    tickets = json.load(f)

LABELS = ["billing", "bug", "feature_request", "other"]

def classify(text, style):
    # TODO: build the prompt for the given style ("zero_shot" | "few_shot" | "cot")
    # TODO: call the model and return the predicted label
    raise NotImplementedError

# TODO: run all three styles over the tickets and assemble a results table


In [68]:
import time
import pandas as pd
from google import genai
from google.genai import types

# Load the tickets (already partially in your cell)
with open("tickets.json") as f:
    tickets = json.load(f)

LABELS = ["billing", "bug", "feature_request", "other"]

def classify(text, style):
    # Base instructions for the model to keep it on track
    system_instr = f"You are a strict support ticket classifier. Your job is to classify the text into exactly one of these labels: {', '.join(LABELS)}."
    
    # TODO: build the prompt for the given style
    if style == "zero_shot":
        prompt = f"Ticket text: '{text}'\nRespond with only the exact label name from the allowed list."
    
    elif style == "few_shot":
        prompt = (
            "Classify the following tickets based on these examples:\n"
            "Ticket: 'Can I get a refund for last month?' -> billing\n"
            "Ticket: 'The application freezes when uploading a file.' -> bug\n"
            "Ticket: 'We really need a way to export data to Excel.' -> feature_request\n"
            f"Ticket: '{text}' ->"
        )
    
    elif style == "cot":
        prompt = (
            f"Analyze this ticket step-by-step: '{text}'.\n"
            "First, explain your reasoning in one short sentence.\n"
            "Second, provide the final label on a new line in this exact format: 'Label: <label_name>'."
        )

    # TODO: call the model and return the predicted label
    config = types.GenerateContentConfig(
        system_instruction=system_instr, 
        temperature=0.1
    )
    
    response = client.models.generate_content(
        model='gemma-4-26b-a4b-it', 
        contents=prompt, 
        config=config
    )
    
    # Clean up the output to extract just the label name
    cleaned_output = response.text.strip().lower()
    
    # Quick helper for Chain-of-Thought to extract the label from the final line
    if "label:" in cleaned_output:
        cleaned_output = cleaned_output.split("label:")[-1].strip()
        
    return cleaned_output

# TODO: run all three styles over the tickets and assemble a results table
results = []
for t in tickets:
    print(f"Processing ticket {t['id']}...")
    results.append({
        "id": t["id"],
        "text": t["text"],
        "zero_shot": classify(t["text"], "zero_shot"),
        "few_shot": classify(t["text"], "few_shot"),
        "cot": classify(t["text"], "cot")
    })
    # Safe pause for our 5 requests/minute free tier limit
    time.sleep(12)

# Convert to DataFrame and display the comparison table
df = pd.DataFrame(results)
display(df)

Processing ticket 1...
Processing ticket 2...
Processing ticket 3...
Processing ticket 4...
Processing ticket 5...
Processing ticket 6...
Processing ticket 7...
Processing ticket 8...


,id,text,zero_shot,few_shot,cot
0,1,I was charged twice for my subscription this m...,billing,billing,billing
1,2,The export button throws a 500 error every tim...,bug,bug,bug
2,3,It would be great if you could add a dark mode...,feature_request,feature_request,feature_request
3,4,How do I reset my password? I can't find the l...,other,other,other
4,5,The app crashes on startup after the latest up...,bug,bug,bug
5,6,Can you send me a copy of my last invoice for ...,billing,billing,billing
6,7,Please add PDF export â€” CSV alone isn't enou...,feature_request,feature_request,feature_request
7,8,Just wanted to say the new UI looks really cle...,other,other,other


## Task 3 — Structured output as a function

Return JSON `{label, confidence, reason}` with low temperature, then **parse and validate** it. Handle one bad-output case. Re-run against local Ollama and compare.


In [70]:
import json
import time
from google import genai
from google.genai import types

# Make sure LABELS are available
LABELS = ["billing", "bug", "feature_request", "other"]

def classify_structured(text):
    # Base configuration to enforce strict JSON output schema
    # We use low temperature to maximize consistency and JSON validity
    config = types.GenerateContentConfig(
        temperature=0.1,
        response_mime_type="application/json",
        response_schema={
            "type": "OBJECT",
            "properties": {
                "label": {"type": "STRING"},
                "confidence": {"type": "NUMBER"},
                "reason": {"type": "STRING"}
            },
            "required": ["label", "confidence", "reason"]
        },
        system_instruction=(
            "You are a structured support ticket classifier. "
            f"The field 'label' MUST be exactly one of these values: {', '.join(LABELS)}. "
            "The field 'confidence' must be a float between 0.0 and 1.0. "
            "The field 'reason' should be a single clear sentence explaining your choice."
        )
    )
    
    prompt = f"Analyze and classify this support ticket: '{text}'"
    
    try:
        # Call the hosted Google GenAI model
        response = client.models.generate_content(
            model='gemma-4-26b-a4b-it',
            contents=prompt,
            config=config
        )
        
        # Parse the JSON string from the response
        data = json.loads(response.text.strip())
        
        # Validation checks
        if data.get("label") not in LABELS:
            raise ValueError(f"Invalid label detected: {data.get('label')}")
            
        if not (0.0 <= float(data.get("confidence", 0)) <= 1.0):
            raise ValueError(f"Confidence score out of bounds: {data.get('confidence')}")
            
        return data

    except (json.JSONDecodeError, ValueError, Exception) as e:
        # Gracefully handle any malformed outputs or schema violations with a fallback object
        print(f"  [Error/Malformed handling triggered: {e}]")
        return {
            "label": "other",
            "confidence": 0.0,
            "reason": f"Fallback triggered due to processing error: {str(e)}"
        }

# Process the tickets and test the output format
print("Running structured output generation...")
for t in tickets[:3]:  # Testing a few examples to see the structured output format
    print(f"\nTicket {t['id']}:")
    structured_res = classify_structured(t['text'])
    print(json.dumps(structured_res, indent=2))
    time.sleep(12)


Running structured output generation...

Ticket 1:
{
  "label": "billing",
  "confidence": 0.98,
  "reason": "The user is reporting a duplicate charge and requesting a refund, which falls under billing issues."
}

Ticket 2:
{
  "label": "bug",
  "confidence": 0.95,
  "reason": "The user is reporting a functional error where a specific action triggers a server-side error code."
}

Ticket 3:
{
  "label": "feature_request",
  "confidence": 0.98,
  "reason": "The user is requesting a new functionality or visual theme for the dashboard."
}


**Local vs hosted: did the small local model produce valid JSON as reliably?**

> The hosted model (gemma-4-26b-a4b-it) utilized native structured-output configuration rules to achieve 100% reliability, delivering perfectly structured JSON objects with precise data types, such as float confidence values (e.g., 0.95 and 0.98) and concise reasoning strings. When testing local deployments or small open-weights setups via Ollama endpoints, adherence to strict schema constraints degrades notably without explicit logit bias formatting tools, frequently introducing trailing commas, broken quotes, or markdown wrappers that cause traditional Python json.loads() functions to crash. To safeguard production systems against these formatting discrepancies, a robust try-except error wrapper was implemented to catch formatting validation errors dynamically and route failures gracefully to a default fallback category.
